# [센서 실습 17] 누수 감지 센서 – 빗물·누수 감지 모듈 (FC-37 + LM393)

물이 닿으면 전기가 통하는 **누수 감지 패드**의 디지털 출력을 읽어 누수를 감지하고, **밸브 자동 차단**과 알림 메시지를 만들며 대응 시간을 측정합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
감지 패드에는 빗살 모양의 전극 두 개가 서로 닿지 않게 놓여 있습니다.
- 물이 전극 사이에 닿으면 전기가 통해 **저항이 크게 낮아짐**
- 비교기 칩(LM393)이 저항 변화를 기준값과 비교해 **디지털 출력**을 냄: **물 감지 시 LOW(0)**
- 가변저항으로 감도 조절, A0 핀으로 아날로그 값도 얻을 수 있음
- 전기가 계속 흐르면 전극이 부식되므로, 정밀한 장치는 측정할 때만 전원을 켬

### 부품 사양
| 항목 | 내용 |
|---|---|
| 출력 | D0 디지털 (물 감지 시 LOW) + A0 아날로그 |
| 전원 | 3.3 ~ 5V |
| 감지 면적 | 약 5cm × 4cm |
| 응답 | 즉시 (물이 닿는 순간) |

## 2. 라즈베리파이 연결 방법
**준비물**: 누수 감지 모듈(패드 + 비교기 보드), 릴레이 모듈, 전자 밸브(선택), 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| 모듈 VCC / GND | 3.3V / GND |  |
| 모듈 D0 | GPIO17 (11번 핀) | 물 감지 시 0 |
| 릴레이 IN | GPIO18 (12번 핀) | 급수 밸브 차단용 |



**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
핀 레벨을 누수 여부로 바꾸는 함수입니다. (액티브 로우)

In [ ]:
def is_leak(pin_level):
    """물이 닿으면 LOW(0)"""
    return int(pin_level == 0)

### 4-2. 실제 센서 클래스
센서 핀과 밸브 릴레이를 함께 다룹니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealLeakSensor:
    def __init__(self, pin=17, valve_pin=18):
        from gpiozero import DigitalInputDevice, OutputDevice
        self.dev = DigitalInputDevice(pin, pull_up=None, active_state=True)
        self.valve = OutputDevice(valve_pin, active_high=False, initial_value=False)

    def read(self):
        level = int(self.dev.value)
        return {"level": level, "leak": is_leak(level), "valve_closed": int(self.valve.value)}

    def close_valve(self):
        self.valve.on()

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 세탁기 아래 바닥, 10분 관찰 (5초 간격) | 가정 누수 |
| 누수 시작 | 200초에 호스 연결부에서 물이 샘 | |
| 물 번짐 | 처음 15초는 물방울이 닿았다 말았다 함 | 패드에 물이 고이기 전 |
| 밸브 차단 효과 | 차단 후 60초 지나면 물이 더 늘지 않고, 닦아 내면(차단 120초 후) 마름 | 대응 상황 |

In [ ]:
class VirtualLeakSensor:
    def __init__(self, dt=5, seed=17):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)
        self.valve_closed_at = None

    def read(self):
        wet = False
        if self.t >= 200:
            wet = self.rng.random() < 0.5 if self.t < 215 else True          # 처음엔 간헐적
        if self.valve_closed_at is not None and self.t > self.valve_closed_at + 120:
            wet = False                                                     # 닦아 내서 마름
        level = 0 if wet else 1
        self.t += self.dt
        return {"level": level, "leak": is_leak(level), "valve_closed": int(self.valve_closed_at is not None)}

    def close_valve(self):
        if self.valve_closed_at is None:
            self.valve_closed_at = self.t

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualLeakSensor(dt=5)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
5초 간격으로 10분 측정합니다. **2회 연속 누수가 감지되면** 반복문 안에서 즉시 밸브를 잠급니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 120               # 측정 횟수
INTERVAL = 5               # 측정 간격(초)

sensor = RealLeakSensor() if USE_REAL_SENSOR else VirtualLeakSensor(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if len(records) >= 2 and records[-1]["leak"] and records[-2]["leak"] and not value["valve_closed"]:
        sensor.close_valve()                 # 2회 연속 감지 → 급수 밸브 차단
        records[-1]["valve_closed"] = 1
        print(f"🚨 {records[-1]['time_s']}초 누수 감지 → 급수 밸브 차단, 관리자에게 알림 전송")
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
누수가 처음 감지된 시각과 밸브를 잠근 시각의 차이(**대응 시간**)를 계산합니다.

In [ ]:
first = df[df["leak"] == 1]["time_s"].min()
closed = df[df["valve_closed"] == 1]["time_s"].min()
dry = df[(df["time_s"] > closed) & (df["leak"] == 0)]["time_s"].min()
print(f"처음 물 감지: {first:.0f}초")
print(f"밸브 차단  : {closed:.0f}초  (대응 시간 {closed - first:.0f}초)")
print(f"바닥 마름  : {dry:.0f}초")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].step(df["time_s"], df["leak"], "k-", where="post"); ax[0].fill_between(df["time_s"], df["leak"], step="post", color="lightgray")
ax[0].set_ylabel("누수 감지"); ax[0].set_title("누수 감지와 밸브 자동 차단")
ax[1].step(df["time_s"], df["valve_closed"], "k-", where="post"); ax[1].set_ylabel("밸브 차단")
ax[1].set_xlabel("시간 (초)")
for a in ax: a.set_yticks([0, 1]); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "누수센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "빗물·누수 감지 모듈 (FC-37 + LM393)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/leak", payload)
# client.disconnect()

## 직접 해 보기
- 밸브 차단 조건을 1회 감지로 바꾸면 어떤 장단점이 있을까요?
- 측정 간격을 1초로 바꾸면 대응 시간이 얼마나 줄어드나요?
- 누수가 감지되면 보낼 MQTT 알림 메시지(JSON)를 직접 설계해 보세요.